REST Resources, Methods and Naming

This becomes easy once you understand this sentence:

> **The URL identifies the thing. The HTTP method says what to do with it. The status code explains the result.**

For example:

```http
GET /v1/books/b42
```

Means:

- **Resource:** book `b42`
- **Method:** retrieve it
- **Expected result:** `200 OK` with the book’s data

That is the foundation. Everything else builds on it.

## 1. Start from absolute zero: What is an API?

An **API** allows one piece of software to communicate with another.

In Harbour:

- The client might be a website or mobile application.
- The server stores information about books, members and loans.
- The client sends an HTTP request.
- The server processes it and sends an HTTP response.

Example:

```http
GET /v1/books/b42
```

Response:

```http
HTTP/1.1 200 OK
Content-Type: application/json
```

```json
{
  "id": "b42",
  "title": "Clean Code",
  "isbn": "9780132350884",
  "status": "available"
}
```

Think of a library desk:

| API concept | Library-desk equivalent |
|---|---|
| URL | Which record or drawer you are referring to |
| HTTP method | What you want the librarian to do |
| Request body | Extra details you supply |
| Status code | Whether the request succeeded or failed |
| Response body | The information or document handed back |

# 2. What does REST mean?

**REST** stands for **Representational State Transfer**.

It is an architectural style for designing networked systems. You do not need to memorise that definition to do your job.

For this week, concentrate on its resource-oriented approach:

1. Important things in the system are treated as **resources**.
2. Each resource has a URL.
3. Standard HTTP methods describe operations on resources.
4. Resources can be represented using formats such as JSON.
5. Requests should contain the information needed to understand them.

A small professional clarification:

> Using resource-based URLs does not automatically make an API completely RESTful. Many APIs called “REST APIs” are more accurately pragmatic, resource-oriented HTTP APIs. Your role focuses on that resource-oriented design.

## 3. What is a resource?

A **resource** is an identifiable thing or concept in your system.

In Harbour, obvious resources are:

- A book
- A member
- A loan

Resources are usually represented by **nouns**, not actions.

Good resource names:

```text
books
members
loans
```

Poor resource names:

```text
getBooks
createMember
borrowBook
returnLoan
deleteMember
```

Those poor examples describe actions. Actions belong primarily in the **HTTP method**, not the URL.

### Resource versus representation

The **resource** is the thing itself: book `b42`.

A **representation** is how that resource is presented in a response:

```json
{
  "id": "b42",
  "title": "Clean Code"
}
```

The same resource could theoretically have JSON, XML or another representation. Your API will probably use JSON.

### A resource is not necessarily a database table

Your database design and API design are related, but they are not identical.

For example, a single `Book` API response could combine information from several database tables. The client should not need to know your database structure.

# 4. Collections and individual resources

Most resource APIs have two important URL patterns.

## A. Collection

A collection contains multiple resources:

```text
/v1/books
/v1/members
/v1/loans
```

For example:

```http
GET /v1/books
```

Means: “Retrieve the book collection.”

## B. Individual item

An individual resource is identified within its collection:

```text
/v1/books/{bookId}
/v1/members/{memberId}
/v1/loans/{loanId}
```

An actual request uses a real identifier:

```http
GET /v1/books/b42
```

Means: “Retrieve book `b42`.”

The pattern is therefore:

```text
/resource-collection       → the collection
/resource-collection/{id}  → one item in that collection
```

# 5. The HTTP methods you must understand

| Method | Main purpose | Typical example | Typical success |
|---|---|---|---|
| `GET` | Retrieve a resource or collection | `GET /v1/books/b42` | `200 OK` |
| `POST` | Create a resource or submit something for processing | `POST /v1/books` | `201 Created` |
| `PUT` | Replace a resource with a complete representation | `PUT /v1/books/b42` | `200 OK` or `204 No Content` |
| `PATCH` | Apply a partial update | `PATCH /v1/books/b42` | `200 OK` or `204 No Content` |
| `DELETE` | Remove a resource | `DELETE /v1/books/b42` | Usually `204 No Content` |

## GET: Read

```http
GET /v1/books
```

or:

```http
GET /v1/books/b42
```

`GET` should not create, update or delete anything.

This is wrong:

```http
GET /v1/books/delete/b42
```

Why? Because `GET` promises clients, browsers and intermediary systems that the request is for retrieval, not destruction.

## POST: Create or submit for processing

To create a book when the server will assign its ID:

```http
POST /v1/books
Content-Type: application/json
```

```json
{
  "title": "The Pragmatic Programmer",
  "isbn": "9780135957059"
}
```

Successful response:

```http
HTTP/1.1 201 Created
Location: /v1/books/b43
```

The `Location` header tells the client where to find the new resource.

Notice that we do **not** need:

```text
POST /v1/books/create
```

`POST` already communicates creation.

`POST` can also be used for operations that do not fit neatly into CRUD, but it should not become your default method for absolutely everything.

## PUT: Complete replacement

`PUT` replaces the editable representation of a resource.

```http
PUT /v1/books/b42
```

```json
{
  "title": "Clean Code",
  "isbn": "9780132350884",
  "publicationYear": 2008
}
```

The important word is **complete**.

If the client omits an editable field, the server may treat that field as removed or reset, according to the contract. Therefore, do not use `PUT` merely to change one field unless the client sends the complete required representation.

Your Harbour contract does not have to support `PUT`. A smaller, well-defined API is better than supporting every method unnecessarily.

## PATCH: Partial update

Use `PATCH` when the client wants to change only part of a resource:

```http
PATCH /v1/books/b42
Content-Type: application/json
```

```json
{
  "title": "Clean Code: Updated Edition"
}
```

The other fields remain unchanged.

### The simplest PUT versus PATCH rule

- **PUT:** “Replace the resource with this complete version.”
- **PATCH:** “Change these particular fields.”

For Harbour, `PATCH` is a sensible choice for ordinary updates.

## DELETE: Remove

```http
DELETE /v1/books/b42
```

A common successful response is:

```http
HTTP/1.1 204 No Content
```

`204` means the operation succeeded, but there is no response body.

Do not automatically provide `DELETE` for every resource. For example, deleting historical loans could damage Harbour’s audit history. It may be better to cancel or close loans through a status change.

# 6. Safe and idempotent: a short but important distinction

Member 3 covers this in greater depth, but you need the basics when choosing methods.

## Safe

A **safe** method is intended only for retrieval, not resource modification.

- `GET` is safe.
- `POST`, `PUT`, `PATCH` and `DELETE` are not formally safe.

Logging a request on the server does not make `GET` unsafe. The point is that the client is not requesting a state change.

## Idempotent

An **idempotent** method has the same intended server-side effect whether the identical request is applied once or repeatedly.

| Method | Safe? | Idempotent? |
|---|---:|---:|
| `GET` | Yes | Yes |
| `POST` | No | Normally no |
| `PUT` | No | Yes |
| `PATCH` | No | Not automatically |
| `DELETE` | No | Yes |

Sending the same `PUT` twice should leave the resource in the same state.

Sending the same `POST /v1/books` twice might create two books.

A useful technical detail:

> PUT is idempotent, but it is not formally “safe.” In everyday speech, people may call repeating PUT “safe”; in HTTP terminology, the precise word is **idempotent**.

Also, idempotency concerns the intended effect, not necessarily identical responses. A repeated `DELETE` could return `204` the first time and `404` the second time, while the resource remains absent.

# 7. Professional resource-naming rules

## Rule 1: Use nouns, not verbs

Good:

```text
GET    /v1/books
POST   /v1/books
DELETE /v1/books/b42
```

Bad:

```text
GET    /v1/getBooks
POST   /v1/createBook
GET    /v1/deleteBook/b42
```

## Rule 2: Use plural names for collections

Use:

```text
/books
/members
/loans
```

Then identify an item:

```text
/books/b42
```

Consistency matters more than arguing endlessly about unusual singular cases.

## Rule 3: Use lowercase and hyphens in multiword paths

Good:

```text
/loan-policies
/library-cards
```

Avoid mixing styles:

```text
/loanPolicies
/library_cards
/LibraryCards
```

JSON field names can separately follow your team’s chosen convention, commonly `camelCase`.

## Rule 4: Use stable identifiers

Good:

```text
/books/b42
```

Less suitable:

```text
/books/clean-code
```

Titles can change and two books may have the same title. IDs are more stable.

## Rule 5: Use query parameters for filtering, sorting and pagination

Examples:

```http
GET /v1/books?status=available
GET /v1/books?search=harbour
GET /v1/loans?memberId=m7&status=active
GET /v1/books?page=2&pageSize=20&sort=title
```

Use the path to identify the resource and the query string to modify the view of a collection.

## Rule 6: Avoid unnecessarily deep nesting

Possible:

```text
/members/m7/loans
```

But often this is more flexible:

```text
/loans?memberId=m7
```

A loan relates to both a member and a book, so treating it as a top-level resource is sensible.

Nesting is useful when parent context is genuinely important. Do not create URLs such as:

```text
/libraries/l1/shelves/s2/books/b42/authors/a9
```

unless every level is truly necessary.

## Rule 7: Be consistent

Choose conventions for:

- Base path, such as `/v1`
- Capitalisation
- Trailing slashes
- Query parameter names
- ID formats
- Error responses

A version prefix such as `/v1` is common, but it is not a REST requirement.

## Rule 8: Do not expose implementation details

Prefer:

```text
/books/b42
```

Avoid URLs tied to implementation, such as:

```text
/database/bookRows/select/42
```

The client should interact with domain concepts, not your database.

# 8. Applying this to Harbour

A simple resource model is:

| Resource | Why it deserves to be a resource |
|---|---|
| Book | It can be listed, created, viewed, updated and possibly removed |
| Member | It has its own identity and lifecycle |
| Loan | It connects a book and member and has its own status, dates and history |

A loan should not merely be hidden inside a member or book. It has its own lifecycle:

```text
active → returned
```

or possibly:

```text
active → cancelled
```

## Borrowing a book

Use the loan collection:

```http
POST /v1/loans
Content-Type: application/json
```

```json
{
  "bookId": "b42",
  "memberId": "m7",
  "dueAt": "2026-11-13"
}
```

Successful response:

```http
HTTP/1.1 201 Created
Location: /v1/loans/l91
```

Why this design works:

- `loans` names the resource being created.
- `POST` communicates creation.
- The body contains the details.
- `201` communicates successful creation.
- `Location` identifies the new loan.

## Returning a book

Returning a loan can be modelled as a state change:

```http
PATCH /v1/loans/l91
Content-Type: application/json
```

```json
{
  "status": "returned"
}
```

The server can set `returnedAt` and update book availability.

This avoids an action-based URL such as:

```text
POST /returnLoan
```

An action endpoint like this is sometimes used in real systems:

```text
POST /v1/loans/l91/return
```

It is not automatically wrong, but it is more RPC-style than resource-oriented. For your exercise, a restricted `PATCH` is a clean solution.

# 9. Friday deliverable: Proposed status-code table

Because your team has not agreed its final API contract yet, treat this as a **strong draft to review**, not as an unchangeable answer.

All paths below are under `/v1`.

| Endpoint | Purpose | Success | Expected failures and reasons |
|---|---|---|---|
| `GET /books` | List or search books | `200`: Returns an array, including an empty array | `400`: Invalid filter, sort or pagination parameter |
| `POST /books` | Create a book | `201`: Book created; include `Location` | `400`: Malformed JSON; `409`: ISBN already exists; `422`: Required fields or values fail validation |
| `GET /books/{bookId}` | Retrieve one book | `200`: Book returned | `404`: The requested book does not exist |
| `PATCH /books/{bookId}` | Partially update a book | `200`: Updated book returned | `400`: Malformed request body; `404`: Book not found; `409`: Update causes an ISBN conflict; `422`: Supplied values are invalid |
| `DELETE /books/{bookId}` | Remove a book | `204`: Deleted, with no response body | `404`: Book not found; `409`: Book cannot be deleted because of an active loan |
| `GET /members` | List or search members | `200`: Returns an array, including an empty array | `400`: Invalid query parameter |
| `POST /members` | Register a member | `201`: Member created; include `Location` | `400`: Malformed JSON; `409`: Email or membership number already exists; `422`: Member data fails validation |
| `GET /members/{memberId}` | Retrieve one member | `200`: Member returned | `404`: Member not found |
| `PATCH /members/{memberId}` | Partially update a member | `200`: Updated member returned | `400`: Malformed body; `404`: Member not found; `409`: Email or membership-number conflict; `422`: Invalid values |
| `DELETE /members/{memberId}` | Remove a member | `204`: Deleted, with no response body | `404`: Member not found; `409`: Member has active loans or another condition preventing deletion |
| `GET /loans` | List loans; filter by member, book or status | `200`: Returns an array, including an empty array | `400`: Invalid status, date, pagination or other query value |
| `POST /loans` | Create a loan | `201`: Loan created; include `Location` | `400`: Malformed JSON; `409`: Book is unavailable or member’s current state prevents borrowing; `422`: Fields are missing or invalid, or referenced IDs are unacceptable under this contract |
| `GET /loans/{loanId}` | Retrieve one loan | `200`: Loan returned | `404`: Loan not found |
| `PATCH /loans/{loanId}` | Update permitted loan details or mark it returned | `200`: Updated loan returned | `400`: Malformed body; `404`: Loan not found; `409`: Requested state transition conflicts with the current state, such as modifying a completed loan; `422`: New values are invalid |

### Contract-wide responses

These can apply across endpoints and do not need repeating in every row:

| Status | When it applies |
|---|---|
| `401 Unauthorized` | Authentication is required or the supplied credentials are invalid |
| `403 Forbidden` | The client is authenticated but is not permitted to perform the operation |
| `405 Method Not Allowed` | The resource exists but does not support that method; for example, `DELETE /loans/l91` if loan deletion is prohibited |
| `415 Unsupported Media Type` | A request body uses an unsupported content type |
| `500 Internal Server Error` | An unexpected server failure occurred; never use this for ordinary client validation errors |

Additional decisions in this proposed contract:

- No `DELETE /loans/{loanId}`: preserve loan history.
- No `PUT` endpoints initially: `PATCH` is sufficient for the planned partial updates.
- An empty collection returns `200` with `[]`, not `404`.
- `404` refers primarily to the target resource in the URL not being found.
- `409` means the request conflicts with the resource’s **current state**.
- `422` means the request is structurally readable but its contents fail defined semantic validation.

Some teams choose `400` for all validation errors or `404` for nonexistent IDs referenced in a request body. Those choices can be workable, but the team must define them consistently. Do not select codes randomly endpoint by endpoint.

# 10. Mistakes you should be able to spot immediately

| Mistake | Why it is wrong | Better approach |
|---|---|---|
| `GET /getBooks` | The URL contains an unnecessary action | `GET /books` |
| `POST /books/create` | `POST` already communicates creation | `POST /books` |
| `GET /deleteBook/b42` | `GET` should not delete | `DELETE /books/b42` |
| Using `POST` for every operation | It throws away useful HTTP semantics | Choose the method matching the operation |
| Using `PUT` for one-field updates without the complete representation | `PUT` means replacement | Use `PATCH` |
| Returning `404` for an empty book list | The collection exists; it simply has no items | Return `200` with `[]` |
| Returning `200` with `{"error":"failed"}` | The status contradicts the result | Use an appropriate `4xx` or `5xx` |
| Creating both `/createLoan` and `POST /loans` | Two competing ways to do the same thing | Keep one canonical creation endpoint |
| Deleting all historical loans | It can destroy useful records | Preserve history and use defined status transitions |
| Choosing codes without reasons | Clients cannot reliably handle responses | Document each code and its meaning |

# 11. A simple design process you can use for any resource

When given a new feature, ask these questions in order:

### Step 1: What is the thing?

Identify the noun.

```text
Book, member, loan, reservation, fine
```

### Step 2: Is it a collection or an item?

```text
/books
/books/b42
```

### Step 3: What operation is needed?

- Read → `GET`
- Create through the collection → `POST`
- Complete replacement → `PUT`
- Partial update → `PATCH`
- Remove → `DELETE`

### Step 4: Is it actually a filter?

```text
GET /loans?status=active
```

Do not create `/activeLoans` unless your team has a strong reason.

### Step 5: What does success look like?

Choose `200`, `201` or `204` deliberately.

### Step 6: What can prevent success?

Consider:

- Malformed input
- Validation failure
- Missing resource
- Current-state conflict
- Authentication or permission
- Unexpected server failure

### Step 7: Write examples

An endpoint is much easier to understand when the contract includes a sample request and response.

# 14. Your professional checklist

Before submitting the Harbour contract, check:

- [ ] Every resource is named with a clear noun.
- [ ] Collection and item URLs follow a consistent pattern.
- [ ] URLs do not contain unnecessary actions such as `get`, `create` or `delete`.
- [ ] Each method matches its HTTP meaning.
- [ ] `PUT` is not being used where `PATCH` is intended.
- [ ] Filtering and pagination use query parameters.
- [ ] Loans have their own resource identity and lifecycle.
- [ ] Empty collections return `200` with an empty array.
- [ ] Creation returns `201` and identifies the new resource.
- [ ] `204` responses contain no body.
- [ ] Validation failures and current-state conflicts are distinguished.
- [ ] Every endpoint has documented success and failure responses.
- [ ] Loan history is not casually deleted.
- [ ] Examples in the documentation match the declared endpoints and status codes.

## The one-minute explanation to memorise

> “In our API, URLs identify resources and HTTP methods describe operations. `/books` is the book collection, while `/books/{bookId}` is one book. `GET` reads, `POST` creates through a collection, `PUT` replaces, `PATCH` partially updates, and `DELETE` removes. We use query parameters for filtering and keep verbs out of resource URLs. Finally, every endpoint documents its success and failure status codes so clients know exactly what happened.”